# MILO — Mi asistente de cocina con Gemini, RAG y LangGraph

## 1. Introducción
He desarrollado MILO para consultar documentos de nutrición general y preparar recetas revisadas.
He separado dos tareas: Gemini explica la información recuperada y Python calcula los nutrientes.
Así puedo comprobar de dónde sale una respuesta y reproducir sus cifras.

En este notebook utilizo los mismos módulos de `src/milo` que utiliza la demostración por terminal.
No implemento un segundo agente. La interfaz Streamlit corresponde a una fase posterior.
MILO no prescribe dietas clínicas ni garantiza ausencia de trazas de alérgenos.

## 2. Librerías y configuración
Instalo las dependencias con `python -m pip install -r requirements.txt` y selecciono ese intérprete.
Guardo mi clave en `.env`, en la raíz, como `GOOGLE_API_KEY=...`. Nunca la imprimo.
Las celdas siguientes realizan llamadas reales y requieren conexión, cuota y documentos locales.
Uso `gemini-3.5-flash-lite`, que respondió en la prueba de integración; el modelo es configurable.
Si una llamada falla, la celda conserva el error: no lo sustituyo por una respuesta de ejemplo.

In [1]:
from collections import Counter
import json
from milo.config import ROOT, Settings
from milo.rag import (load_documents, create_embeddings, create_model, message_text,
                      index_chunks, NutritionRAG, show_answer)
from milo.agent import MiloAgent, show_turn
from milo.models import UserProfile, DietaryRestriction

settings = Settings()
assert settings.require_key()  # Compruebo presencia sin mostrar el valor.
print("Modelo:", settings.model)
print("Embeddings:", settings.embedding_model)
respuesta_api = message_text(create_model(settings).invoke("Explica RAG en una frase en español."))
assert respuesta_api.strip()
print("Llamada real a Gemini:", respuesta_api)

Modelo: gemini-3.5-flash-lite
Embeddings: gemini-embedding-001


Llamada real a Gemini: El RAG es una técnica que mejora las respuestas de la Inteligencia Artificial conectándola a documentos externos y actualizados para que no tenga que depender solo de su memoria.


## 3. Documentos de conocimiento
Mi corpus contiene el PDF oficial de AESAN (19 páginas), mi recetario con siete recetas revisadas y
mi guía de técnicas y sustituciones. Los dos documentos propios incluyen referencias y límites.
No atribuyo mis recetas a AESAN ni a USDA. El diccionario técnico de USDA explica los datos y no
forma parte del conocimiento nutricional del RAG.

Mantengo el origen y el hash SHA-256 de cada archivo en `data/sources.json`. El catálogo de 70 alimentos
y el recetario JSON sirven para calcular; no reemplazan a los documentos que indexo.

In [2]:
fuentes = json.loads((ROOT / "data/sources.json").read_text(encoding="utf-8"))
for fuente in fuentes:
    if fuente["usage"] == "rag_general":
        print(fuente["source_id"], "—", fuente["title"])
        print("Archivo:", fuente["path"], "| SHA-256:", fuente["sha256"])
        print("Origen:", fuente["url"], "\n")

AESAN_2022 — Recomendaciones dietéticas saludables y sostenibles (AESAN)
Archivo: data/raw/RECOMENDACIONES_DIETETICAS.pdf | SHA-256: 8a7027defb7395a4b9136487dd9493bb9ff5a43940aafb8ab832dac78b21ccc8
Origen: https://www.aesan.gob.es/AECOSAN/docs/documentos/nutricion/RECOMENDACIONES_DIETETICAS.pdf 

MILO_RECETAS — Mi recetario revisado para MILO
Archivo: data/documents/recetas_revisadas.md | SHA-256: 2351de7a628912e6926b9c39f3d1f8282d1f7a6c60450fc004971d13af3f7802
Origen: data/documents/recetas_revisadas.md 

MILO_TECNICAS — Técnicas culinarias y sustituciones para MILO
Archivo: data/documents/tecnicas_y_sustituciones.md | SHA-256: 61310fce4d950e5eee9b40b349507346d70aa104440a7d38e9f40e1cefcb98b4
Origen: data/documents/tecnicas_y_sustituciones.md 



## 4. Chunking
Divido los documentos en fragmentos de hasta 400 palabras con un solapamiento de 50 cuando una
sección ocupa varias ventanas. Conservo páginas del PDF y secciones de los Markdown. Agrupo títulos
muy cortos del PDF para evitar vectores que contengan solo unas palabras.
Los identificadores dependen del contenido y del hash de la fuente: un cambio produce una colección nueva.

In [3]:
fragmentos = load_documents()
conteo = Counter(f.source_id for f in fragmentos)
print("Fragmentos por documento:", dict(conteo))
print("Total:", len(fragmentos))
assert len(conteo) == 3
assert all(len(f.text.split()) <= 400 for f in fragmentos)
ejemplo = next(f for f in fragmentos if f.source_id == "MILO_RECETAS" and "r001" in f.section)
print("\nEjemplo:", ejemplo.section, "\n", ejemplo.text)

Fragmentos por documento: {'AESAN_2022': 39, 'MILO_RECETAS': 8, 'MILO_TECNICAS': 11}
Total: 58

Ejemplo: r001 — Bowl de pollo y arroz 
 ## r001 — Bowl de pollo y arroz He revisado esta receta para 1 ración. Tiempo orientativo: 20 minutos. Para solicitar sus macros utilizo el ID r001. Ingredientes y cantidades: - Pechuga de pollo sin piel ni hueso, cruda: 160 g; registro fdc_2646170. - Arroz blanco de grano largo sin enriquecer, seco: 80 g; registro fdc_2512381. - Tomate roma, crudo: 120 g; registro fdc_1999634. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 10 g; registro fdc_171413. Preparación: 1. Pesar el arroz en seco y la pechuga cruda, sin piel ni hueso. Lavar y cortar el tomate. 2. Cocer el arroz en agua siguiendo el tiempo del envase; escurrir si sobra agua. 3. Cortar el pollo en tiras. Calentar el aceite medido en una sartén y cocinar el pollo hasta alcanzar 74 °C en el centro, medidos con termómetro. 4. Servir el arroz con el pollo y el tomate. Incorporar el ace

## 5. Embeddings de Gemini
Un embedding representa un texto mediante una lista de números. Uso `gemini-embedding-001`
con 3072 dimensiones tanto para los documentos como para las preguntas.
Esta celda llama realmente a la API; comprobar solo que existe la clave no demostraría su funcionamiento.

In [4]:
embeddings = create_embeddings(settings)
vector = embeddings.embed_query("Receta sencilla con pollo y arroz")
assert len(vector) == 3072 and any(v != 0 for v in vector)
print("Dimensiones del vector recibido:", len(vector))

Dimensiones del vector recibido: 3072


## 6. ChromaDB y recuperación
Guardo los vectores en `storage/chroma`. La indexación añade solo identificadores que faltan,
por lo que repetir esta celda con el mismo corpus no duplica documentos. La colección incorpora
una huella del corpus y del modelo, evitando mezclar versiones.
Después busco por similitud, envío los fragmentos a Gemini y compruebo que use citas existentes.
Las distancias de Chroma sirven para ordenar resultados; no son porcentajes de confianza.

In [5]:
indice = index_chunks(fragmentos, settings)
assert len(indice.get()["ids"]) == len(fragmentos)
print("Vectores guardados:", len(indice.get()["ids"]))
rag = NutritionRAG(settings)  # Abro de nuevo la colección persistida.
pregunta = "¿Qué recomendaciones nutricionales hay para una dieta equilibrada?"
resultado = rag.ask(pregunta)
assert any(f["metadata"]["source_id"] == "AESAN_2022" for f in resultado.fragments)
show_answer(resultado)

Vectores guardados: 58



PREGUNTA: ¿Qué recomendaciones nutricionales hay para una dieta equilibrada?

[1] AESAN_2022 — página 2
2 QUÉ SON LAS RECOMENDACIONES Y PARA QUÉ SIRVEN 3 ALGUNAS IDEAS PRÁCTICAS Al comprar Al cocinar 16 Al comer Muévete más y siéntate menos Para saber más 17 Y LA ACTIVIDAD FÍSICA, POR QUÉ ES BENEFICIOSA PARA LA SALUD ¿Cuánta actividad física debo hacer? 13 Recomendaciones según los grupos de edad 14 QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE Más hortalizas y frutas Patatas y otros tubérculos con moderación 4 Cereales, mejor integrales 5 Más legumbres 6 Más frutos secos 7 Más pescado, mejor azul 8 Huevos con moderación, según tus necesidades 9 Leche y productos lácteos con moderación, según tus necesidades Menos carne, mejor si es blanca 10 Aceite de oliva cada día Agua del grifo siempre 11 Reduce e incluso evita tomar… 12

[2] AESAN_2022 — página 18
AGUA El agua es la bebida de elección en una dieta saludable. Bebe agua siempre que tengas sed. Las necesidades pueden a

In [6]:
resultado_receta = rag.ask("Recomiéndame una receta sencilla con pollo y arroz.")
assert any("r001" in f["metadata"].get("section", "") for f in resultado_receta.fragments)
show_answer(resultado_receta)


PREGUNTA: Recomiéndame una receta sencilla con pollo y arroz.

[1] MILO_RECETAS — r001 — Bowl de pollo y arroz
## r001 — Bowl de pollo y arroz He revisado esta receta para 1 ración. Tiempo orientativo: 20 minutos. Para solicitar sus macros utilizo el ID r001. Ingredientes y cantidades: - Pechuga de pollo sin piel ni hueso, cruda: 160 g; registro fdc_2646170. - Arroz blanco de grano largo sin enriquecer, seco: 80 g; registro fdc_2512381. - Tomate roma, crudo: 120 g; registro fdc_1999634. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 10 g; registro fdc_171413. Preparación: 1. Pesar el arroz en seco y la pechuga cruda, sin piel ni hueso. Lavar y cortar el tomate. 2. Cocer el arroz en agua siguiendo el tiempo del envase; escurrir si sobra agua. 3. Cortar el pollo en tiras. Calentar el aceite medido en una sartén y cocinar el pollo hasta alcanzar 74 °C en el centro, medidos con termómetro. 4. Servir el arroz con el pollo y el tomate. Incorporar el aceite y jugos de la sartén

## 7. Agente LangGraph
He utilizado cuatro nodos sin ciclos: **interpretar → recuperar → herramientas → responder**.
Gemini transforma la petición en una intención estructurada y puede resolver referencias al historial.
El nodo de recuperación utiliza el mismo RAG. Python valida la receta contra el perfil y calcula
sus nutrientes antes de responder. Si la receta está pendiente o es incompatible, detengo su recomendación.

Para una receta, Gemini explica brevemente la técnica y la ficha numérica se muestra directamente
desde Python. El ajuste es un escalado uniforme de cantidades, no una optimización ni una prescripción.

In [7]:
agente = MiloAgent(rag)
print("Nodos:", " → ".join(["interpretar", "recuperar", "herramientas", "responder"]))
print("Recetas del catálogo:", len(agente.recipes))
print("Recetas revisadas:", sum(r.review_status == "verified" for r in agente.recipes.values()))

Nodos: interpretar → recuperar → herramientas → responder
Recetas del catálogo: 12
Recetas revisadas: 7


## 8. Memoria conversacional
Uso `InMemorySaver` y un `thread_id` por conversación. El estado conserva mensajes, perfil y última
receta validada. Esta memoria dura mientras vive el objeto Python; no sobrevive al reinicio del proceso.
Registro las restricciones explícitamente en `UserProfile` para que el modelo no pueda eliminarlas.
La extracción automática de un perfil completo desde texto libre sigue fuera de esta versión.

En el ejemplo registro una dieta vegetariana, pido una tortilla y después pregunto por «esa receta».
Finalmente intento pedir pollo con el mismo perfil. Compruebo también que otro hilo no herede el historial.

In [8]:
perfil = UserProfile(user_id="demo_vegetariano", restrictions=[
    DietaryRestriction(kind="diet", value="vegetarian")
])
turno1 = agente.ask("Quiero la receta r002 de tortilla de espinacas.",
                    thread_id="memoria", profile=perfil)
show_turn(turno1)
assert turno1["tool_result"] is not None

turno2 = agente.ask("¿Cuántas calorías y proteínas tiene esa receta?", thread_id="memoria")
show_turn(turno2)
assert turno2["last_recipe_id"] == "r002"
assert turno2["tool_result"]["per_serving"] == turno1["tool_result"]["per_serving"]
assert len(turno2["messages"]) == 4
assert turno2["profile"] == perfil.model_dump(mode="json")

Ruta: recipe | Herramienta calculada: True

PREGUNTA: Quiero la receta r002 de tortilla de espinacas.

[1] MILO_RECETAS — r002 — Tortilla de espinacas
## r002 — Tortilla de espinacas He revisado esta receta para 1 ración. Tiempo orientativo: 15 minutos. Para solicitar sus macros utilizo el ID r002. Ingredientes y cantidades: - Huevo entero grande, crudo, sin cáscara: 120 g; registro fdc_748967. - Espinaca baby, cruda: 80 g; registro fdc_1999632. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 8 g; registro fdc_171413. Preparación: 1. Pesar el huevo sin cáscara. Lavar las espinacas y secarlas. 2. Calentar el aceite medido en una sartén antiadherente y saltear las espinacas hasta que pierdan volumen. 3. Batir el huevo, añadirlo y cuajar la tortilla por ambos lados; comprobar 71 °C en el centro. 4. Servir la tortilla completa como una ración. Alérgenos conocidos: egg. Esto no garantiza ausencia de trazas ni sustituye la etiqueta del producto. He conservado el estado de pesado

Ruta: nutrition | Herramienta calculada: True

PREGUNTA: ¿Cuántas calorías y proteínas tiene esa receta?

[1] MILO_RECETAS — r002 — Tortilla de espinacas
## r002 — Tortilla de espinacas He revisado esta receta para 1 ración. Tiempo orientativo: 15 minutos. Para solicitar sus macros utilizo el ID r002. Ingredientes y cantidades: - Huevo entero grande, crudo, sin cáscara: 120 g; registro fdc_748967. - Espinaca baby, cruda: 80 g; registro fdc_1999632. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 8 g; registro fdc_171413. Preparación: 1. Pesar el huevo sin cáscara. Lavar las espinacas y secarlas. 2. Calentar el aceite medido en una sartén antiadherente y saltear las espinacas hasta que pierdan volumen. 3. Batir el huevo, añadirlo y cuajar la tortilla por ambos lados; comprobar 71 °C en el centro. 4. Servir la tortilla completa como una ración. Alérgenos conocidos: egg. Esto no garantiza ausencia de trazas ni sustituye la etiqueta del producto. He conservado el estado de pes

In [9]:
turno3 = agente.ask("Quiero la receta r001 de pollo con arroz.", thread_id="memoria")
show_turn(turno3)
assert turno3["blocked"] and turno3["tool_result"] is None
otro_hilo = agente.graph.get_state({"configurable": {"thread_id": "otra_persona"}}).values
assert not otro_hilo
print("Memoria comprobada: referencia anterior, perfil conservado y otro hilo vacío.")

Ruta: clarify | Herramienta calculada: False

PREGUNTA: Quiero la receta r001 de pollo con arroz.

RESPUESTA DEL AGENTE:
El perfil del usuario tiene restricciones vegetarianas, pero la receta r001 (Bowl de pollo y arroz) contiene carne. Por favor, confirma si deseas mantener la restricción vegetariana o actualizar tu UserProfile.
Memoria comprobada: referencia anterior, perfil conservado y otro hilo vacío.


## 9. Preguntas de demostración
Ejecuto cinco consultas del agente y muestro pregunta, fragmentos, procedencia y respuesta.
Compruebo si aparece la fuente esperada entre los resultados recuperados. Es una evaluación pequeña
de recuperación y de rutas; no demuestra que cualquier respuesta posible sea correcta.
La validez formal de las citas tampoco garantiza por sí sola que cada frase esté respaldada: reviso
su correspondencia leyendo los fragmentos que aparecen junto a la respuesta.

In [10]:
casos = [
    ("¿Qué recomendaciones nutricionales hay para una dieta equilibrada?", "AESAN_2022", None),
    ("Recomiéndame una receta sencilla con pollo y arroz.", "MILO_RECETAS", "r001"),
    ("Calcula las calorías y proteínas de la receta r001.", "MILO_RECETAS", "r001"),
    ("¿Puedo sustituir arroz seco por el mismo peso de patata cruda?", "MILO_TECNICAS", None),
    ("¿Cómo preparo una tortilla de espinacas?", "MILO_RECETAS", "r002"),
]
aciertos = 0
for numero, (pregunta, fuente_esperada, seccion) in enumerate(casos, 1):
    turno = agente.ask(pregunta, thread_id=f"evaluacion_{numero}")
    show_turn(turno)
    encontrado = any(f["metadata"]["source_id"] == fuente_esperada and
        (seccion is None or seccion in f["metadata"].get("section", "")) for f in turno["fragments"])
    assert encontrado and not turno["blocked"]
    aciertos += int(encontrado)
print(f"\nFuentes esperadas recuperadas: {aciertos}/{len(casos)}")

Ruta: general | Herramienta calculada: False

PREGUNTA: ¿Qué recomendaciones nutricionales hay para una dieta equilibrada?

[1] AESAN_2022 — página 2
2 QUÉ SON LAS RECOMENDACIONES Y PARA QUÉ SIRVEN 3 ALGUNAS IDEAS PRÁCTICAS Al comprar Al cocinar 16 Al comer Muévete más y siéntate menos Para saber más 17 Y LA ACTIVIDAD FÍSICA, POR QUÉ ES BENEFICIOSA PARA LA SALUD ¿Cuánta actividad física debo hacer? 13 Recomendaciones según los grupos de edad 14 QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE Más hortalizas y frutas Patatas y otros tubérculos con moderación 4 Cereales, mejor integrales 5 Más legumbres 6 Más frutos secos 7 Más pescado, mejor azul 8 Huevos con moderación, según tus necesidades 9 Leche y productos lácteos con moderación, según tus necesidades Menos carne, mejor si es blanca 10 Aceite de oliva cada día Agua del grifo siempre 11 Reduce e incluso evita tomar… 12

[2] AESAN_2022 — página 18
AGUA El agua es la bebida de elección en una dieta saludable. Bebe agua sie

Ruta: recipe | Herramienta calculada: True

PREGUNTA: Recomiéndame una receta sencilla con pollo y arroz.

[1] MILO_RECETAS — r001 — Bowl de pollo y arroz
## r001 — Bowl de pollo y arroz He revisado esta receta para 1 ración. Tiempo orientativo: 20 minutos. Para solicitar sus macros utilizo el ID r001. Ingredientes y cantidades: - Pechuga de pollo sin piel ni hueso, cruda: 160 g; registro fdc_2646170. - Arroz blanco de grano largo sin enriquecer, seco: 80 g; registro fdc_2512381. - Tomate roma, crudo: 120 g; registro fdc_1999634. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 10 g; registro fdc_171413. Preparación: 1. Pesar el arroz en seco y la pechuga cruda, sin piel ni hueso. Lavar y cortar el tomate. 2. Cocer el arroz en agua siguiendo el tiempo del envase; escurrir si sobra agua. 3. Cortar el pollo en tiras. Calentar el aceite medido en una sartén y cocinar el pollo hasta alcanzar 74 °C en el centro, medidos con termómetro. 4. Servir el arroz con el pollo y el tomate

Ruta: nutrition | Herramienta calculada: True

PREGUNTA: Calcula las calorías y proteínas de la receta r001.

[1] MILO_RECETAS — r001 — Bowl de pollo y arroz
## r001 — Bowl de pollo y arroz He revisado esta receta para 1 ración. Tiempo orientativo: 20 minutos. Para solicitar sus macros utilizo el ID r001. Ingredientes y cantidades: - Pechuga de pollo sin piel ni hueso, cruda: 160 g; registro fdc_2646170. - Arroz blanco de grano largo sin enriquecer, seco: 80 g; registro fdc_2512381. - Tomate roma, crudo: 120 g; registro fdc_1999634. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 10 g; registro fdc_171413. Preparación: 1. Pesar el arroz en seco y la pechuga cruda, sin piel ni hueso. Lavar y cortar el tomate. 2. Cocer el arroz en agua siguiendo el tiempo del envase; escurrir si sobra agua. 3. Cortar el pollo en tiras. Calentar el aceite medido en una sartén y cocinar el pollo hasta alcanzar 74 °C en el centro, medidos con termómetro. 4. Servir el arroz con el pollo y el tom

Ruta: general | Herramienta calculada: False

PREGUNTA: ¿Puedo sustituir arroz seco por el mismo peso de patata cruda?

[1] MILO_TECNICAS — Sustituir arroz, pasta, quinoa o patata
## Sustituir arroz, pasta, quinoa o patata Estos alimentos pueden ocupar un papel parecido como acompañamiento, pero difieren en humedad, preparación y composición. No los cambio gramo por gramo ni afirmo que aporten las mismas calorías. Elijo el registro del sustituto, fijo su cantidad y estado y recalculo la receta con Python. Si cambio arroz seco por patata cruda, también cambia la técnica y el volumen. Un escalado matemático no demuestra que la adaptación sea culinariamente adecuada. Ante restricciones de gluten reviso el producto concreto: una pasta de trigo no se vuelve compatible por aparecer en una lista genérica de sustituciones.

[2] MILO_TECNICAS — Preparar y pesar los ingredientes
## Preparar y pesar los ingredientes Antes de cocinar, leo qué ingredientes se pesan crudos, secos, cocinados o escurr

Ruta: recipe | Herramienta calculada: True

PREGUNTA: ¿Cómo preparo una tortilla de espinacas?

[1] MILO_RECETAS — r002 — Tortilla de espinacas
## r002 — Tortilla de espinacas He revisado esta receta para 1 ración. Tiempo orientativo: 15 minutos. Para solicitar sus macros utilizo el ID r002. Ingredientes y cantidades: - Huevo entero grande, crudo, sin cáscara: 120 g; registro fdc_748967. - Espinaca baby, cruda: 80 g; registro fdc_1999632. - Aceite de oliva para ensalada o cocina (genérico USDA SR): 8 g; registro fdc_171413. Preparación: 1. Pesar el huevo sin cáscara. Lavar las espinacas y secarlas. 2. Calentar el aceite medido en una sartén antiadherente y saltear las espinacas hasta que pierdan volumen. 3. Batir el huevo, añadirlo y cuajar la tortilla por ambos lados; comprobar 71 °C en el centro. 4. Servir la tortilla completa como una ración. Alérgenos conocidos: egg. Esto no garantiza ausencia de trazas ni sustituye la etiqueta del producto. He conservado el estado de pesado de cad

## 10. Conclusiones
He integrado documentos reales, embeddings de Gemini, persistencia en Chroma, respuestas con citas,
un grafo sencillo y memoria entre turnos. Las comprobaciones de las celdas distinguen una ejecución
correcta de un archivo que simplemente contiene código. Los pesos y nutrientes proceden del catálogo
y de funciones que también pruebo con `pytest`.

He conservado 70 alimentos y 12 recetas; siete están revisadas y cinco siguen pendientes de composición
o alérgenos de productos compuestos. Un dato ausente sigue siendo desconocido. No estimo rendimiento
de cocción ni doy por equivalentes los pesos crudos y cocinados.

Me queda ampliar la evaluación, revisar las cinco recetas pendientes y preparar la presentación.
Streamlit, planificación semanal, lista de compra y modo nevera siguen pendientes. Mi memoria actual
es local al proceso y mis controles de citas no son un verificador automático de verdad.
El estado detallado está en `PROJECT_CHECKLIST.md`; las instrucciones reproducibles están en `README.md`.